<h1 id ="libraries" style="color:#E36149;">Libraries</h1>

In [ ]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

#Matplotlib Config
plt.style.use('fivethirtyeight')

plt.rcParams['font.size'] = 16
plt.rcParams['axes.labelsize'] = 20
# plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 24
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.titlesize'] = 20
width, height = plt.figaspect(1.68)
fig = plt.figure(figsize=(width,height), dpi=400)

In [ ]:
df = pd.read_csv('/kaggle/input/employee-future-prediction/Employee.csv')

In [ ]:
df.shape

In [ ]:
df.info()

<h1 id="eda" style="color:#E36149;">EDA</h1>

In [ ]:
df.isna().sum()

**No missing values**

In [ ]:
duplicates = df.duplicated().sum()
df = df.drop_duplicates()

print('No. of duplicate records :',duplicates)
print('Shape after dropping duplicate records :',df.shape)

In [ ]:
df.describe()

In [ ]:
f, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, mask=np.zeros_like(corr, dtype=np.bool),
            cmap=sns.diverging_palette(220, 10, as_cmap=True),
            square=True, ax=ax ,annot=True)

* **'PaymentTier' & 'Age' have a very weak negative correlation with Our Target variable ('LeaveOrNot')**
* **'JoiningYear' has a very weak positive correlation with Target variable ('LeaveOrNot')**

In [ ]:
df['JoiningYear'] = df['JoiningYear'].astype('object')
sns.countplot(data = df ,x='JoiningYear',hue='LeaveOrNot')

**Surprising ! Majority of the employees joined in 2018 (recently) have left the company**

In [ ]:
sns.countplot(data = df ,x='EverBenched',hue='LeaveOrNot')

**Its surprising that Employees being Benched on working projects have not left the company 😂 jokes apart ..Back to EDA**

In [ ]:
sns.countplot(data = df ,x='ExperienceInCurrentDomain',hue='LeaveOrNot')

* **There is not much data available where Employees having Experience in their Current Domain Greater than 5 Years**
* **As the experience of Employees increases they chose to work/stay with company**

In [ ]:
sns.countplot(data = df ,x='Gender',hue='LeaveOrNot')

* **Majority of the Male Employees choose not to leave the company**
* **While the ratio ( Leave / Not Leave ) of the Female Employees is  almost equal to 1**

In [ ]:
df['PaymentTier'] = df['PaymentTier'].astype('category')
sns.countplot(data = df ,x='PaymentTier',hue='LeaveOrNot')

* **Most of the Employees are having payment Tier 3**
* **By observing the trend it seems that 'PaymentTier' Catergory is an Ordinal Variable**
* **Where, Tier 3 > Tier 2 > Tier 1**

In [ ]:
sns.countplot(data = df ,x='City',hue='LeaveOrNot')

*  **Majority of the Employees residing / Working in Bangalore & New Delhi chose not to leave the company**
* **Maybe the work culture in Bangalore & New Delhi is pretty good when compared to Pune**


In [ ]:
sns.countplot(data = df ,x='Education',hue='LeaveOrNot')

* **Majority of the Employees who pursued Bachelors are working for these companies**
* **Education can be considered as Ordinal variable where  PHD > Masters > Bachelors**

In [ ]:
groups = ['Young', 'MiddleAged', 'Adulthood']
df['AgeGroup'] = pd.qcut(df['Age'], q=3, labels=groups)
sns.countplot(data = df ,x='AgeGroup',hue='LeaveOrNot')

* **From the bar chart , it is clear that Young Employees left their jobs more than the MiddleAged & Adulthood Employees.**
* **Maybe young employees who leave want to pursue their dreams**

<h1 id="feature"style="color:#E36149;">Feature Engineering</h1>

In [ ]:
X = df.drop('LeaveOrNot',axis=1)
y= df.LeaveOrNot.values

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [ ]:
#new feature engineering code which got  me lgbm's acc : 83% and precision 97%

multi_categories = ['AgeGroup','EverBenched','City','JoiningYear']
ordinal_cat = [['PHD','Masters','Bachelors'],[1,2,3]]
binary_categories=['Gender','EverBenched']
transformer = ColumnTransformer(transformers=[('ohe1', OneHotEncoder(sparse='False'), multi_categories),
                                             ('oe',OrdinalEncoder(categories=ordinal_cat),['Education','PaymentTier']),
                                             ('ohe2', OneHotEncoder(drop='first',sparse='False'), binary_categories)],remainder='passthrough')

# old feat engineering code which got me lgbm's acc : 83% and precision 93% 

# df1= df.copy()
# df1['PaymentTier'] = df['PaymentTier'].astype(np.uint8)
# df1['LeaveOrNot'] = df['LeaveOrNot'].astype('category')
# dummy_col = ['JoiningYear','City','AgeGroup']
# df1=pd.get_dummies(df, columns=dummy_col,prefix = ['year','city','age'])
# df1["is_masters"] = df1["Education"].map(lambda is_masters: 1 if is_masters == "Masters" else 0).astype(np.uint8)
# df1["is_male"] = df1["Gender"].map(lambda is_male: 1 if is_male == "Male" else 0).astype(np.uint8)
# df1["EverBenched"] = df1['EverBenched'].map(lambda is_benched: 1 if is_benched =='Yes' else 0).astype(np.uint8)
# df1['Senority'] = df1[['ExperienceInCurrentDomain','Age']].apply(lambda x: 1 if x.ExperienceInCurrentDomain >= 3 else 0, axis=1).astype(np.uint8) # 0 : junior , 1: senior
# # df1['isRecentEmployee']= df['JoiningYear'].map(lambda x : 1 if (x == '2018'or'2017') else 0).astype(np.uint8)
# df1=df1.drop(['Education','Gender'],axis=1)
# df1.head()

X_train = transformer.fit_transform(X_train)
X_train.shape
X_test = transformer.transform(X_test)

<h1 id="model" style="color:#E36149;">Building Models..</h1>

In [ ]:
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth=7,random_state=2)
lrc = LogisticRegression(solver='liblinear', penalty='l1')
rfc = RandomForestClassifier(n_estimators=17, random_state=2,max_depth=5)
abc = AdaBoostClassifier(n_estimators=17, random_state=2,learning_rate=0.2)
bc = BaggingClassifier(n_estimators=17, random_state=2)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)
gbdt = GradientBoostingClassifier(n_estimators=18,random_state=2)
xgb = XGBClassifier(n_estimators=17,random_state=2,use_label_encoder=False,eval_metric='mlogloss')
lgbm= LGBMClassifier(verbose=-1,
                          learning_rate=0.1,
                          max_depth=6,
                          num_leaves=10, 
                          n_estimators=17,
                          max_bin=500,random_state=2)


clfs = {
    'KN' : knc, 
    'NB': mnb, 
    'DT': dtc, 
    'LR': lrc, 
    'RF': rfc, 
    'AdaBoost': abc, 
    'BgC': bc, 
    'ETC': etc,
    'GBDT':gbdt,
    'xgb':xgb,
    'lgbm':lgbm
}

def train_classifier(clf,X_train,y_train,X_test,y_test):
    clf.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred,zero_division=0)
    
    return accuracy,precision

accuracy_scores = []
precision_scores = []

for name,clf in clfs.items():
    
    current_accuracy,current_precision = train_classifier(clf, X_train,y_train,X_test,y_test)
    
#     print("For ",name)
#     print("Accuracy - ",current_accuracy)
#     print("Precision - ",current_precision)
    
    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)
    

performance_df = pd.DataFrame({'Algorithm':clfs.keys(),'Accuracy':accuracy_scores,'Precision':precision_scores}).sort_values('Precision',ascending=False)
performance_df

**Best Working Model is LGBM
 with Accuracy of 83% and Precision of 97%**

<h2 style="color:#E36149;">
<br><br>
 Feedbacks and Suggestions for improving models are welcome 😃</h2>

